# **Energy Demand Forecasting - Datasets Cleaning**


(remark: raw datasets are stored in this project's GitHub repository under data/raw/, cloned at the top of this notebook; cleaned outputs are saved to data/processed/)

#

**Primiary dataset:**

Energy demand historical dataset:

historic_demand_2009_2024.csv

(source: https://www.kaggle.com/datasets/albertovidalrod/electricity-consumption-uk-20092022)


**Supplementary datasets**

(1) Weather (daily) dataset:

all_weather_data.parquet

(source: https://www.kaggle.com/datasets/jakewright/2m-daily-weather-history-uk)

(remark: 2021 populations by region are used as weights for aggregate national weather data)

(source: https://www.nomisweb.co.uk/datasets/pestsyoala)

(Office for National Statistics, "Total population" dataset, via NOMIS — published 26/09/2025)

file: region_location_lookup.csv (file was summarised from the source data)




(2) PMI dataset:

united-kingdom.markit-manufacturing-pmi.csv

(source: https://www.mql5.com/en/economic-calendar/united-kingdom/markit-manufacturing-pmi)


(3) RSI dataset:

retail-sales-index-time-series-v45-filtered-2026-06-30T15-15-28Z.csv

(source: https://www.ons.gov.uk/datasets/retail-sales-index/editions/time-series/versions/45)

(4) Covid Tracker dataset:

OxCGRT_compact_national_v1.csv

(source: https://github.com/OxCGRT/covid-policy-dataset/tree/main)


(Remark: all raw dataset files are stored in github https://raw.githubusercontent.com/paulmcchan/Surrey-AI-and-Sustainability/main/data/raw)


In [ ]:
# Read raw data directly from GitHub (public repo, no auth needed);
# save cleaned outputs to a local Colab folder for this session.
import os
import pandas as pd

RAW_DIR = "https://raw.githubusercontent.com/paulmcchan/Surrey-AI-and-Sustainability/main/data/raw"
PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)


**Cleaning for RSI dataset**

In [ ]:
# RSI dataset cleaning

from google.colab import files
import pandas as pd
import io

# 1. Load your downloaded ONS file
df_rsi = pd.read_csv(f'{RAW_DIR}/retail-sales-index-time-series-v45-filtered-2026-06-30T15-15-28Z.csv')

# 2. Keep only the columns we care about: Date, the Metric Name, and the Value
df_cleaned = df_rsi[['Time', 'Prices', 'v4_1']].copy()
df_cleaned.rename(columns={'Time': 'date', 'v4_1': 'value'}, inplace=True)

# 3. Pivot the table so that the metrics become individual columns
df_pivoted = df_cleaned.pivot(index='date', columns='Prices', values='value').reset_index()

# 4. Clean up the column names for your ML pipeline
df_pivoted.columns.name = None
df_pivoted.columns = ['date', 'rsi_mom_change', 'rsi_volume']

# 5. Convert the 'date' column to proper datetime format (from 'Jan-09' to '2009-01-01')
df_pivoted['date'] = pd.to_datetime(df_pivoted['date'], format='%b-%y')
df_pivoted = df_pivoted.sort_values('date').reset_index(drop=True)

print(df_pivoted.head())

        date  rsi_mom_change  rsi_volume
0 2009-01-01             0.8        87.1
1 2009-02-01            -2.0        85.4
2 2009-03-01             0.5        85.8
3 2009-04-01             0.5        86.2
4 2009-05-01             0.1        86.3


In [ ]:
# Output of cleaned RSI dataset

from google.colab import files

# 1. Save the dataframe to a new CSV file inside Colab
# Set index=False so it doesn't save the row numbers (0, 1, 2...) as a column
new_filename = f'{PROCESSED_DIR}/RSI_Cleaned_2009_2024.csv'
df_pivoted.to_csv(new_filename, index=False)

print(f"File successfully saved as: {new_filename}")


File successfully saved as: data/processed/RSI_Cleaned_2009_2024.csv


**Cleaning of PMI dataset**

In [ ]:
# Load and clean PMI dataset

import pandas as pd
from google.colab import files


# 1. Load the dataset (using tab separation)
df_pmi = pd.read_csv(f'{RAW_DIR}/united-kingdom.markit-manufacturing-pmi.csv', sep='\t')

# 2. Keep only Date and the Actual PMI value
df_pmi = df_pmi[['Date', 'ActualValue']].copy()
df_pmi.rename(columns={'Date': 'date', 'ActualValue': 'pmi'}, inplace=True)

# 3. Convert to proper datetime format
df_pmi['date'] = pd.to_datetime(df_pmi['date'], format='%Y.%m.%d')

# 4. Filter out the mid-month "Flash" readings (keep only day 1 to 5)
df_pmi = df_pmi[df_pmi['date'].dt.day <= 5]

# 5. STRICT TIMEFRAME FILTER: Keep only 2009 to 2024 inclusive
df_pmi = df_pmi[(df_pmi['date'] >= '2009-01-01') & (df_pmi['date'] <= '2024-12-31')]

# 6. Sort chronologically and reset index
df_pmi = df_pmi.sort_values('date').reset_index(drop=True)

print("Cleaned and Filtered PMI Data (2009-2024):")
print(f"Total Rows: {len(df_pmi)}")
print(df_pmi.head(5))
print(df_pmi.tail(5))

# 7. Save the clean file
new_filename = f'{PROCESSED_DIR}/PMI_Cleaned_2009_2024.csv'
df_pmi.to_csv(new_filename, index=False)



Cleaned and Filtered PMI Data (2009-2024):
Total Rows: 192
        date   pmi
0 2009-01-02  34.9
1 2009-02-02  35.8
2 2009-03-02  34.7
3 2009-04-01  39.5
4 2009-05-01  43.1
          date   pmi
187 2024-08-01  52.1
188 2024-09-02  52.5
189 2024-10-01  51.5
190 2024-11-01  49.9
191 2024-12-02  48.0


**Cleaning of Covid Tracker dataset**

In [ ]:
# Load and clean Covid Tracker dataset

import pandas as pd
from google.colab import files

# 1. Load the dataset (using tab separation)
df_covid = pd.read_csv(f'{RAW_DIR}/OxCGRT_compact_national_v1.csv')

# 2. Filter only CountryName == "United Kingdom"
df_covid = df_covid[df_covid['CountryName'] == 'United Kingdom'].copy()

# 3. Keep only Date and StringencyIndex_Average
df_covid_cleaned = df_covid[['Date', 'StringencyIndex_Average']].copy()

# 4. Change date format to yy-mm-dd
df_covid_cleaned['Date'] = pd.to_datetime(df_covid_cleaned['Date'], format='%Y%m%d')

# 4b. Rename column to lowercase 'date' for consistency with weather/PMI/RSI files
df_covid_cleaned = df_covid_cleaned.rename(columns={'Date': 'date'})

# 5. Sort chronologically and reset index
df_covid_cleaned = df_covid_cleaned.sort_values('date').reset_index(drop=True)

# 6. Print

print("Cleaned and Filtered Covide Tracker for UK:")
print(f"Total Rows: {len(df_covid_cleaned)}")
print(df_covid_cleaned.head(5))
print(df_covid_cleaned.tail(5))

# 7. Save the clean file
new_filename = f'{PROCESSED_DIR}/Covid_Cleaned_UK.csv'
df_covid_cleaned.to_csv(new_filename, index=False)



Cleaned and Filtered Covide Tracker for UK:
Total Rows: 1096
        date  StringencyIndex_Average
0 2020-01-01                      0.0
1 2020-01-02                      0.0
2 2020-01-03                      0.0
3 2020-01-04                      0.0
4 2020-01-05                      0.0
           date  StringencyIndex_Average
1091 2022-12-27                     5.56
1092 2022-12-28                     5.56
1093 2022-12-29                     5.56
1094 2022-12-30                     5.56
1095 2022-12-31                     5.56


**Cleaning for Weather dataset**
1. Great Britain (GB) is divided into 11 regions with populations (2021-6-30) as weights - South East, London, North West, East of England, West Midlands, South West, Yorkshire and The Humber, Scotland, Wales, North East


2. Select 11 weather stations to represent 11 regions - Crawley, London, Manchester, Norwich, Birmingham, Merehead, Leeds, Bishopbriggs, Nuneaton, Mynachdy, Newcastle

3. Filter the 11 locations weather data.  Add Region names and weights (populations)

4. Export the cleaned weather dataset


In [ ]:
# Load and clean Weather dataset

# 1. Load and clean Weather dataset
import pandas as pd
from google.colab import files

df_weather = pd.read_parquet(f'{RAW_DIR}/all_weather_data.parquet')
df_weather['location'] = df_weather['location'].str.strip()

# 2. Load the lookup table
df_lookup = pd.read_csv(f'{RAW_DIR}/region_location_lookup.csv')

# 3. Filter weather data to only the 11 representative locations
df_weather = df_weather[df_weather['location'].isin(df_lookup['location'])]

# 4. Merge region + weights_population onto every matching row
df_weather = df_weather.merge(df_lookup, on='location', how='left')

# 5. Drop wind_direction and wind_direction_numerical, as these are irrelevant when aggregating the weather parameters.
df_weather = df_weather.drop(columns=['wind_direction', 'wind_direction_numerical'])

# 6. Sanity check — should both be 0
print(df_weather['region'].isna().sum())
print(df_weather['weights_population'].isna().sum())

print(df_weather.head(11))
print(df_weather.tail(11))

# 7. Save the clean file
new_filename = f'{PROCESSED_DIR}/all_weather_data_cleaned_UK.csv'
df_weather.to_csv(new_filename, index=False)




0
0
        location        date  min_temp °c  max_temp °c  rain mm  humidity %  \
0     Manchester  2009-01-01         -5.0          0.0      0.0        79.0   
1     Birmingham  2009-01-01         -5.0          0.0      0.0        88.0   
2       Mynachdy  2009-01-01         -3.0          2.0      0.0        72.0   
3       Nuneaton  2009-01-01         -5.0          0.0      0.0        89.0   
4       Merehead  2009-01-01         -4.0          0.0      0.0        76.0   
5   Bishopbriggs  2009-01-01         -4.0          1.0      0.0        89.0   
6        Norwich  2009-01-01          1.0          4.0      0.0        86.0   
7      Newcastle  2009-01-01         -2.0          3.0      0.0        89.0   
8        Crawley  2009-01-01         -2.0          3.0      0.0        91.0   
9         London  2009-01-01         -2.0          2.0      0.0        93.0   
10         Leeds  2009-01-01         -5.0          0.0      0.0        87.0   

    cloud_cover %  wind_speed km/h             

In [ ]:
import pandas as pd

df_weather = pd.read_csv(f'{PROCESSED_DIR}/all_weather_data_cleaned_UK.csv')
df_weather['date'] = pd.to_datetime(df_weather['date'])

weather_cols = ['min_temp °c', 'max_temp °c', 'rain mm', 'humidity %', 'cloud_cover %', 'wind_speed km/h']

# 1. Drop the one incomplete day (2024-06-12, only 3/11 locations present)
loc_count = df_weather.groupby('date')['location'].transform('count')
df_weather = df_weather[loc_count == 11]

# 2. Multiply each weather value by its location's population weight
for col in weather_cols:
    df_weather[col + '_w'] = df_weather[col] * df_weather['weights_population']

# 3. Group by date: sum the weighted values and sum the weights
agg_dict = {col + '_w': 'sum' for col in weather_cols}
agg_dict['weights_population'] = 'sum'

daily_weather = df_weather.groupby('date').agg(agg_dict).reset_index()

# 4. Divide weighted sum by total weight to get the population-weighted average
for col in weather_cols:
    daily_weather[col] = daily_weather[col + '_w'] / daily_weather['weights_population']
    daily_weather = daily_weather.drop(columns=[col + '_w'])

daily_weather = daily_weather.drop(columns=['weights_population'])

print(daily_weather.shape)
print(daily_weather.head())
print(daily_weather.tail())

# 5. Output the clean aggregated national dataset
new_filename = f'{PROCESSED_DIR}/daily_weather_national_cleaned.csv'
daily_weather.to_csv(new_filename, index=False)



(5637, 7)
        date  min_temp °c  max_temp °c   rain mm  humidity %  cloud_cover %  \
0 2009-01-01    -3.191595     1.390157  0.000000   86.180682      34.502431   
1 2009-01-02    -1.165751     3.982250  0.057535   88.748310      42.802863   
2 2009-01-03    -2.924495     2.265193  0.000000   82.751066      15.203297   
3 2009-01-04    -2.504722     1.102831  0.259393   83.714745      58.949315   
4 2009-01-05    -2.194277     2.385938  1.140070   85.845601      38.274472   

   wind_speed km/h  
0         7.647022  
1        12.022057  
2         9.317808  
3        12.398495  
4        16.322209  
           date  min_temp °c  max_temp °c   rain mm  humidity %  \
5632 2024-06-07     5.942214    15.979762  0.945449   76.386028   
5633 2024-06-08     7.498590    15.718072  0.673082   72.936995   
5634 2024-06-09     6.500877    14.277135  2.046141   77.670240   
5635 2024-06-10     7.116516    13.908054  5.163640   78.696970   
5636 2024-06-11     6.068854    14.058734  2.011847   

**Missing data in weather dataset**

The merged csv, merged_energy_demand_dataset.csv file has 5641 rows.

But, the weather dataset, daily_weather_national_cleaned.csv, has 5637 rows.

The gaps, 4 rows, are due to missing 29 Feb in every leap year, 2012, 2016, 2020 and 2024


In [ ]:
# Fixing missing data for 29 Feburary in leap years in weather dataset, daily_weather_national_cleaned.csv

import pandas as pd

# 1. Load the weather dataset
df_weather = pd.read_csv(f'{PROCESSED_DIR}/daily_weather_national_cleaned.csv')
df_weather['date'] = pd.to_datetime(df_weather['date'])

# 2. Build a complete daily calendar spanning the full range, and reindex onto it
full_calendar = pd.date_range(df_weather['date'].min(), df_weather['date'].max(), freq='D')
df_weather = df_weather.set_index('date').reindex(full_calendar)
df_weather.index.name = 'date'

# 3. Confirm which dates were missing (should be the 4 leap days)
missing_dates = df_weather[df_weather.isna().any(axis=1)].index
print("Dates filled via interpolation:")
print(missing_dates)

# 4. Linearly interpolate the missing rows (single isolated gaps, safe to interpolate)
weather_cols = ['min_temp °c', 'max_temp °c', 'rain mm', 'humidity %', 'cloud_cover %', 'wind_speed km/h']
df_weather[weather_cols] = df_weather[weather_cols].interpolate(method='linear')

df_weather = df_weather.reset_index()

# 5. Sanity checks
print(df_weather.shape)
print(df_weather.isna().sum())
print(df_weather[df_weather['date'].isin(missing_dates)])

# 6. Save the corrected file
df_weather.to_csv(f'{PROCESSED_DIR}/daily_weather_national_cleaned.csv', index=False)

Dates filled via interpolation:
DatetimeIndex(['2012-02-29', '2016-02-29', '2020-02-29', '2024-02-29'], dtype='datetime64[ns]', name='date', freq='1461D')
(5641, 7)
date               0
min_temp °c        0
max_temp °c        0
rain mm            0
humidity %         0
cloud_cover %      0
wind_speed km/h    0
dtype: int64
           date  min_temp °c  max_temp °c   rain mm  humidity %  \
1154 2012-02-29     6.148657    13.009232  0.020795   90.047836   
2615 2016-02-29     2.294675     8.862655  3.860354   81.834706   
4076 2020-02-29     1.925490     7.803024  5.062234   77.837630   
5537 2024-02-29     3.016231     8.002530  5.165479   86.986273   

      cloud_cover %  wind_speed km/h  
1154      51.793007         9.563117  
2615      64.972003        23.261214  
4076      74.029745        23.470711  
5537      82.871723        18.312384  


**Merging supplementary datasets to primary dataset (historic_demand_2009_2024.csv)**

1.  Merge by "date"
2.  Granularity -> daily

**Note:**
1.  Primary dataset granularity - half an hour, 2009-01-01 00:30 to 2024-12-05 24:00
2.  Weather dataset - daily, 2009-01-01 to 2024-06-11
3.  PMI dataset - monthly, 2009-01 to 2024-12
4.  PSI dataset - monthly, 2009-01 to 2024-12
5.  COVID Tracker dataset - daily, 2020-01-01 to 2022-12-31



In [ ]:
# Load primary dataset (historic_demand_2009_2024.csv) and aggregate to daily demand

import pandas as pd

# 1. Load and aggregate demand to daily
df_demand = pd.read_csv(f'{RAW_DIR}/historic_demand_2009_2024.csv')
df_demand['settlement_date'] = pd.to_datetime(df_demand['settlement_date'])

daily_demand = df_demand.groupby('settlement_date').agg(
    nd_total_mwh=('nd', lambda x: x.sum() * 0.5),      # total national demand energy (MWh)
    is_holiday=('is_holiday', 'max')                    # constant across a day, max just extracts it
).reset_index()

daily_demand = daily_demand.rename(columns={'settlement_date': 'date'})

# 2. Load weather (already daily, already population-weighted)
df_weather = pd.read_csv(f'{PROCESSED_DIR}/daily_weather_national_cleaned.csv')
df_weather['date'] = pd.to_datetime(df_weather['date'])

merged = daily_demand.merge(df_weather, on='date', how='left')

# 3. Load PMI and RSI, merge via year-month (not exact date)
df_pmi = pd.read_csv(f'{PROCESSED_DIR}/PMI_Cleaned_2009_2024.csv')
df_pmi['date'] = pd.to_datetime(df_pmi['date'])
df_pmi['year_month'] = df_pmi['date'].dt.to_period('M')

df_rsi = pd.read_csv(f'{PROCESSED_DIR}/RSI_Cleaned_2009_2024.csv')
df_rsi['date'] = pd.to_datetime(df_rsi['date'])
df_rsi['year_month'] = df_rsi['date'].dt.to_period('M')

merged['year_month'] = merged['date'].dt.to_period('M')
merged = merged.merge(df_pmi[['year_month', 'pmi']], on='year_month', how='left')
merged = merged.merge(df_rsi[['year_month', 'rsi_mom_change', 'rsi_volume']], on='year_month', how='left')
merged = merged.drop(columns=['year_month'])

# 4. Load COVID stringency, merge, fill gaps outside tracked window with 0
df_covid = pd.read_csv(f'{PROCESSED_DIR}/Covid_Cleaned_UK.csv')
df_covid['date'] = pd.to_datetime(df_covid['date'])

merged = merged.merge(df_covid, on='date', how='left')
merged['StringencyIndex_Average'] = merged['StringencyIndex_Average'].fillna(0)

# 5. Trim to weather's coverage limit
merged = merged[merged['date'] <= '2024-06-11'].reset_index(drop=True)

# 6. Sanity checks
print(merged.shape)
print(merged.isna().sum())
print(merged.head())
print(merged.tail())

merged.to_csv(f'{PROCESSED_DIR}/merged_energy_demand_dataset.csv', index=False)


(5641, 13)
date                       0
nd_total_mwh               0
is_holiday                 0
min_temp °c                0
max_temp °c                0
rain mm                    0
humidity %                 0
cloud_cover %              0
wind_speed km/h            0
pmi                        0
rsi_mom_change             0
rsi_volume                 0
StringencyIndex_Average    0
dtype: int64
        date  nd_total_mwh  is_holiday  min_temp °c  max_temp °c   rain mm  \
0 2009-01-01      894660.5           1    -3.191595     1.390157  0.000000   
1 2009-01-02      960360.5           0    -1.165751     3.982250  0.057535   
2 2009-01-03      948845.5           0    -2.924495     2.265193  0.000000   
3 2009-01-04      955703.5           0    -2.504722     1.102831  0.259393   
4 2009-01-05     1090823.0           0    -2.194277     2.385938  1.140070   

   humidity %  cloud_cover %  wind_speed km/h   pmi  rsi_mom_change  \
0   86.180682      34.502431         7.647022  34.9        